# Agentic NAS Workflow: Step-by-Step Exploration
Welcome to the interactive exploration of our Nextcloud ingestion and deduplication pipeline! In this notebook, we will walk through the core logic powering our automated NAS workflow.

In [1]:
# Setup: Import our custom modules from the `src` package  # noqa: EXE002
import os
import sys
from pathlib import Path

import numpy as np

repo_root = Path.cwd().parent.resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

# Ensure the project root is in the Python path
sys.path.append(os.path.abspath('..'))

try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except NameError:
    pass

print("Modules loaded successfully!")

Modules loaded successfully!


## Step 1: Data Ingestion and Mock File Generation
First, let's explore our local ingestion directory. This is where files land before they are processed by the pipeline. We will create some mock files (including an exact duplicate) to demonstrate our pipeline's capabilities.

In [3]:
DUMMY_INGEST_DIR = "/home/coder/projects/agentic-nas-workflow/data/mock/mock_gdrive"

In [4]:
# Let's see what files we have in the ingest directory
if not os.path.exists(DUMMY_INGEST_DIR):
    print(f"Directory {DUMMY_INGEST_DIR} does not exist yet. Let's create some dummy files for this demo.")
    os.makedirs(DUMMY_INGEST_DIR, exist_ok=True)
    
    with open(os.path.join(DUMMY_INGEST_DIR, "report.pdf"), "w") as f:
        f.write("Dummy PDF content")
    with open(os.path.join(DUMMY_INGEST_DIR, "photo.jpg"), "w") as f:
        f.write("Dummy Image content")
    # A duplicate file
    with open(os.path.join(DUMMY_INGEST_DIR, "report_copy.pdf"), "w") as f:
        f.write("Dummy PDF content")

files_to_process = []
for root, _, files in os.walk(DUMMY_INGEST_DIR):
    for file in files:
        files_to_process.append(os.path.join(root, file))

print(f"Found {len(files_to_process)} files to process:")
for f in files_to_process:
    print(f" - {f}")

Found 5 files to process:
 - /home/coder/projects/agentic-nas-workflow/data/mock/mock_gdrive/report.pdf
 - /home/coder/projects/agentic-nas-workflow/data/mock/mock_gdrive/report_copy.pdf
 - /home/coder/projects/agentic-nas-workflow/data/mock/mock_gdrive/photo.jpg
 - /home/coder/projects/agentic-nas-workflow/data/mock/mock_gdrive/aws_invoice_july.txt
 - /home/coder/projects/agentic-nas-workflow/data/mock/mock_gdrive/tax_2026.pdf


## Step 2: Cryptographic Deduplication (BLAKE3 Hashing)
To prevent duplicate files from eating up precious NAS/Nextcloud space, we use **BLAKE3 Hashing** for exact bitwise deduplication.
Unlike md5 or sha256, BLAKE3 is extremely fast and can stream large files efficiently in chunks.

Let's compute the hashes for our three generated files and verify that our duplicate detector successfully flags the matching PDF files while letting the unique image pass.

In [5]:
from src.elt.scanner import calculate_blake3
from src.vector.multimodal import _get_text_model


/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Compute and compare BLAKE3 hashes
pdf_path = os.path.join(DUMMY_INGEST_DIR, "report.pdf")
copy_path = os.path.join(DUMMY_INGEST_DIR, "report_copy.pdf")
photo_path = os.path.join(DUMMY_INGEST_DIR, "photo.jpg")

hash_pdf = calculate_blake3(pdf_path)
hash_copy = calculate_blake3(copy_path)
hash_photo = calculate_blake3(photo_path)

print(f"report.pdf hash:       {hash_pdf}")
print(f"report_copy.pdf hash:  {hash_copy}")
print(f"photo.jpg hash:        {hash_photo}\n")

# Assert and show results
print(f"Does report.pdf match report_copy.pdf? {hash_pdf == hash_copy} (Expected: True)")
print(f"Does report.pdf match photo.jpg?        {hash_pdf == hash_photo} (Expected: False)")

assert hash_pdf == hash_copy, "Error: PDF copy should have the same hash!"
assert hash_pdf != hash_photo, "Error: Image and PDF should have different hashes!"
print("\nSuccess! Cryptographic deduplication is working perfectly.")

report.pdf hash:       f9ea87c1bcce628cc6260f8d832dabe258857fd2513980bccf263554db65d5fe
report_copy.pdf hash:  f9ea87c1bcce628cc6260f8d832dabe258857fd2513980bccf263554db65d5fe
photo.jpg hash:        53f34198c1152cadab64c5745d9ecff100a9f1cf0aa9610d2cbebdd01ba98bcf

Does report.pdf match report_copy.pdf? True (Expected: True)
Does report.pdf match photo.jpg?        False (Expected: False)

Success! Cryptographic deduplication is working perfectly.


## Step 3: Semantic Deduplication (Content Similarity)
Cryptographic deduplication is perfect for finding **exact, byte-for-byte identical files**. However, it fails if a file is slightly modified (e.g., a PDF report with a single updated typo or a draft and a final version of a text file).

This is where **Semantic Deduplication** comes in. By analyzing the textual or visual content of files, we can calculate a **similarity score** and identify files that are nearly identical.

In [7]:
text_model = _get_text_model()

def calculate_cosine_similarity(text1: str, text2: str) -> float:
    """Calculates cosine similarity using v5 Dense Vectors."""
    # Generate the 384-dimensional dense vectors
    vecs = list(text_model.embed([text1, text2]))
    a, b = vecs[0], vecs[1]
    
    # Mathematical cosine similarity: (A dot B) / (||A|| * ||B||)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def calculate_jaccard_similarity(text1: str, text2: str) -> float:
    """Calculates basic lexical overlap (Jaccard)."""
    set1 = set(text1.lower().split())
    set2 = set(text2.lower().split())
    return len(set1.intersection(set2)) / len(set1.union(set2))

Loading FastEmbed Text Model (BAAI/bge-small-en-v1.5)...


Fetching 5 files: 100%|██████████| 5/5 [00:21<00:00,  4.24s/it]


In [9]:
# Let's define three texts to compare
doc_original = "The agentic NAS workflow automates our file ingestion, cryptographic hashing, and remote uploads to Nextcloud."
doc_near_duplicate = "The agentic NAS workflow automatically handles our file ingestion, cryptographic hashing, and remote uploads to Nextcloud server."
doc_completely_different = "Tomorrow's weather forecast predicts clear skies with a mild breeze and temperatures around 72 degrees."

print("Document Comparisons:")
print("-" * 50)

# Compare original vs near duplicate
jaccard_near = calculate_jaccard_similarity(doc_original, doc_near_duplicate)
cosine_near = calculate_cosine_similarity(doc_original, doc_near_duplicate)
print("Original vs Near Duplicate:")
print(f"  - Jaccard Similarity: {jaccard_near:.4f}")
print(f"  - Cosine Similarity:  {cosine_near:.4f}")

# Compare original vs different
jaccard_diff = calculate_jaccard_similarity(doc_original, doc_completely_different)
cosine_diff = calculate_cosine_similarity(doc_original, doc_completely_different)
print("\nOriginal vs Completely Different:")
print(f"  - Jaccard Similarity: {jaccard_diff:.4f}")
print(f"  - Cosine Similarity:  {cosine_diff:.4f}")

# Assert similarity behaves as expected
assert cosine_near > 0.80, "Error: Near duplicates should have high similarity!"
assert cosine_diff < 0.15, "Error: Completely different documents should have low similarity!"
print("\nSuccess! Semantic deduplication detects near-duplicates perfectly.")

Document Comparisons:
--------------------------------------------------
Original vs Near Duplicate:
  - Jaccard Similarity: 0.6842
  - Cosine Similarity:  0.9700

Original vs Completely Different:
  - Jaccard Similarity: 0.0345
  - Cosine Similarity:  0.3668


AssertionError: Error: Completely different documents should have low similarity!

## Step 4: WebDAV Storage and Prefect Orchestration
Once our files are ingested and checked for duplicates, they are routed and uploaded idempotently to **Nextcloud** using WebDAV (`webdav4`).

The entire process is orchestrated as a pipeline flow using **Prefect 3.0**, with task retries and exponential backoffs configured to handle network hiccups seamlessly.

Let's check out our WebDAV upload helper and run the complete pipeline!
*(Note: To make this exploration safe and robust, if your Nextcloud instance or Prefect server is not currently reachable, the cells will catch the connection exceptions and describe the simulated production behavior instead of crashing!)*

In [2]:
# Test uploading a single file to Nextcloud WebDAV
from prefect.blocks.system import Secret
from webdav4.client import Client

from src.configs import load_settings

settings = load_settings()
INGEST_DIR = settings["ingestion"]["cloud_sources"]["google_drive"]
NEXTCLOUD_URL = settings["nextcloud"]["url"]

In [3]:
print(f"Targeting Nextcloud at: {NEXTCLOUD_URL}")
print(f"Scanning Ingest Directory: {INGEST_DIR}")

Targeting Nextcloud at: http://192.168.1.55:30027/remote.php/webdav
Scanning Ingest Directory: /cloud_ingest/gdrive_ahabib9387


### Step 4.1: Initiate WebDAV Client

In [4]:
# 1. Fetch secrets safely inside Jupyter's async loop
user_block = await Secret.load("nextcloud-username")
pass_block = await Secret.load("nextcloud-password")

# 2. Instantiate the WebDAV client (This is our Dependency!)
client = Client(NEXTCLOUD_URL, auth=(user_block.get(), pass_block.get()))

print("\nWebDAV Client successfully instantiated.")
print("Root Nextcloud Folders:")
for item in client.ls("/"):
    print(f" - {item['name']}")


WebDAV Client successfully instantiated.
Root Nextcloud Folders:
 - Photos
 - Documents
 - Templates


### Step 4.2: Test Ingestion with Dummy Data

In [12]:
remote_test_dir = "/Documents/DummyIngestionTest"

# Idempotently create the remote folder if it doesn't exist
if not client.exists(remote_test_dir):
    client.mkdir(remote_test_dir)
    print(f"Created remote folder: {remote_test_dir}")

if not os.path.exists(DUMMY_INGEST_DIR):
    print(f"CRITICAL ERROR: Could not find local folder at {DUMMY_INGEST_DIR}")
else:
    print(f"Scanning local directory: {DUMMY_INGEST_DIR}...\n")
    
    for filename in os.listdir(DUMMY_INGEST_DIR):
        local_path = os.path.join(DUMMY_INGEST_DIR, filename)
        
        # SRE Check: Ensure we are only uploading files, not sub-directories
        if os.path.isfile(local_path):
            remote_path = f"{remote_test_dir}/{filename}"
            print(f" ➔ Uploading: {filename}...")
            
            # Using upload_file with from_path and to_path
            client.upload_file(from_path=local_path, to_path=remote_path, overwrite=True)
            print(f"    [Success] Uploaded to {remote_path}")

# Verify the State Database (Nextcloud)
print(f"\nVerification - Files currently in Nextcloud {remote_test_dir}:")
for item in client.ls(remote_test_dir):
    # We filter out the directory itself and just show the files
    if item['type'] == 'file':
        # Convert bytes to MB for easier reading
        size_mb = item.get('size', 0) / (1024 * 1024)
        print(f" - {item['name']} ({size_mb:.2f} MB)")

Scanning local directory: /home/coder/projects/agentic-nas-workflow/data/mock/mock_gdrive...

 ➔ Uploading: report.pdf...
    [Success] Uploaded to /Documents/DummyIngestionTest/report.pdf
 ➔ Uploading: report_copy.pdf...
    [Success] Uploaded to /Documents/DummyIngestionTest/report_copy.pdf
 ➔ Uploading: photo.jpg...
    [Success] Uploaded to /Documents/DummyIngestionTest/photo.jpg
 ➔ Uploading: aws_invoice_july.txt...
    [Success] Uploaded to /Documents/DummyIngestionTest/aws_invoice_july.txt
 ➔ Uploading: tax_2026.pdf...
    [Success] Uploaded to /Documents/DummyIngestionTest/tax_2026.pdf

Verification - Files currently in Nextcloud /Documents/DummyIngestionTest:
 - Documents/DummyIngestionTest/report.pdf (0.00 MB)
 - Documents/DummyIngestionTest/report_copy.pdf (0.00 MB)
 - Documents/DummyIngestionTest/photo.jpg (0.00 MB)
 - Documents/DummyIngestionTest/aws_invoice_july.txt (0.00 MB)
 - Documents/DummyIngestionTest/tax_2026.pdf (0.00 MB)


### Step 4.3: Test without Prefect Pipeline (manual loop)

In [1]:
import os
import shutil
import sqlite3
import sys
from pathlib import Path

import pandas as pd

repo_root = Path.cwd().parent.resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from openai import OpenAI
from prefect.blocks.system import Secret
from webdav4.client import Client

from src.configs import load_settings, root_path
from src.elt.ingestion import execute_ingestion, trigger_nextcloud_occ_scan
from src.elt.scanner import calculate_blake3, scan_sources
from src.elt.strategy import generate_strategy
from src.state.schema import init_schema, reconcile_deletions


In [8]:
# 1. Setup Paths and DB
settings = load_settings()
db_path = repo_root / "data/test_ledger.db"
nextcloud_mount = Path("/nextcloud_data")
user_files_dir = nextcloud_mount / "admin" / "files"

print("="*60)
print("🧹 STARTING PHYSICAL DISK CONSOLIDATION & PURGE")
print("="*60)

# 2. Step 1: Scan all physical files on disk and group by BLAKE3 Hash
IGNORE_DIRS = [] # Nextcloud default stock directories
hash_map = {} # {hash: [list_of_physical_paths]}

for root, _, files in os.walk(user_files_dir):
    for file in files:
        local_path = os.path.join(root, file)
        rel_path = "/" + os.path.relpath(local_path, user_files_dir).replace("\\", "/")
        
        # Skip Nextcloud stock templates
        if any(rel_path.lower().startswith(d) for d in IGNORE_DIRS):
            continue
            
        file_hash = calculate_blake3(local_path)
        if file_hash not in hash_map:
            hash_map[file_hash] = []
        hash_map[file_hash].append(local_path)

print(f"📊 Discovered {len(hash_map)} unique physical contents across disk.\n")

# 3. Step 2: Ask the LLM Agent where each unique file TRULY belongs
llm_key = await Secret.load(settings["llm"]["secret_blocks"]["gemini_api_key"])
llm_client = OpenAI(api_key=llm_key.get(), base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

# Build a temporary staging list for the LLM strategy engine
temp_staging = []
for idx, (f_hash, paths) in enumerate(hash_map.items()):
    primary_path = paths[0]
    filename = os.path.basename(primary_path)
    temp_staging.append({
        "staging_id": idx + 1,
        "source_table": "disk_consolidation",
        "original_path": primary_path,
        "filename": filename
    })

# Ask LLM for the optimal Nextcloud taxonomy path
prompt = f"""
You are an expert Data Architect. We are consolidating a cluttered NAS.
For each unique file, assign it to its ideal, stable Nextcloud directory path.

APPROVED TAXONOMY:
- /Documents/Financial/Invoices
- /Documents/Financial/Taxes
- /Documents/Work_Projects
- /Documents/Reports
- /Media/Photos/Family

FILES TO ROUTE:
{temp_staging}
"""

🧹 STARTING PHYSICAL DISK CONSOLIDATION & PURGE
📊 Discovered 0 unique physical contents across disk.



In [13]:
import instructor
from pydantic import BaseModel


class FileRoute(BaseModel):
    staging_id: int
    proposed_path: str

class Strategy(BaseModel):
    routings: list[FileRoute]

instructor_client = instructor.from_openai(llm_client)
strategy = instructor_client.chat.completions.create(
    model=settings["llm"]["model"],
    response_model=Strategy,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.1
)

route_dict = {r.staging_id: r.proposed_path for r in strategy.routings}

# 4. Step 3: Execute Physical Moves and Delete Duplicates
# Reset production inventory table
conn = sqlite3.connect(db_path)
conn.execute("DELETE FROM production_inventory;")
conn.commit()

purged_duplicates_count = 0
consolidated_count = 0

for idx, (f_hash, paths) in enumerate(hash_map.items()):
    primary_path = paths[0]
    target_rel_path = route_dict.get(idx + 1, f"/Documents/Unsorted/{os.path.basename(primary_path)}").lstrip("/")
    target_abs_path = user_files_dir / target_rel_path
    
    print(f"\n📦 Processing Content Hash [{f_hash[:8]}...]:")
    print(f"   Target Path: /{target_rel_path}")
    
    # Ensure target directory exists
    target_abs_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Move ONE primary copy to the target location
    if Path(primary_path).resolve() != target_abs_path.resolve():
        if target_abs_path.exists():
            print("   🛑 Target already exists!")
        else:
            shutil.move(primary_path, target_abs_path)
            print(f"   ✅ Moved primary file -> /{target_rel_path}")
    else:
        print("   ✅ Primary file already at target location.")
        
    consolidated_count += 1
    
    # Record in production_inventory
    file_size = os.path.getsize(target_abs_path)

    conn.execute("""
        INSERT INTO production_inventory (blake3_hash, nextcloud_path, file_size)
        VALUES (?, ?, ?)
        ON CONFLICT(blake3_hash) DO UPDATE SET 
            nextcloud_path = excluded.nextcloud_path,
            file_size = excluded.file_size
    """, (f_hash, f"/{target_rel_path}", file_size))
    
    # DELETE ALL OTHER PHYSICAL DUPLICATES FROM DISK
    for dup_path in paths[1:]:
        if os.path.exists(dup_path) and Path(dup_path).resolve() != target_abs_path.resolve():
            os.remove(dup_path)
            print(f"   🔥 PURGED physical duplicate: {dup_path}")
            purged_duplicates_count += 1

conn.commit()
conn.close()

# 5. Step 4: Prune Empty Folders
print("\n🧹 Pruning empty directories...")
for root, dirs, files in os.walk(user_files_dir, topdown=False):
    for d in dirs:
        dir_path = os.path.join(root, d)
        if any(dir_path.lower().startswith(str(user_files_dir / ignore.strip('/'))) for ignore in IGNORE_DIRS):
            continue
        try:
            if not os.listdir(dir_path):
                os.rmdir(dir_path)
                print(f"   Removed empty dir: {dir_path}")
        except Exception:  # noqa: BLE001, S110
            pass

print("\n" + "="*60)
print("🎉 CONSOLIDATION COMPLETE!")
print(f"   - Unique Files Consolidated: {consolidated_count}")
print(f"   - Physical Duplicates Purged from Disk: {purged_duplicates_count}")
print("="*60)

print("--- TESTING PREFECT/NOTEBOOK OCC SCAN TRIGGER ---")

# Trigger the scan directly from Python
success = trigger_nextcloud_occ_scan(container_name="ix-nextcloud-nextcloud-1")

if success:
    print("\n🎉 Refresh your Nextcloud Web UI (http://192.168.1.55:30027). All new files are now visible!")


🧹 Pruning empty directories...

🎉 CONSOLIDATION COMPLETE!
   - Unique Files Consolidated: 0
   - Physical Duplicates Purged from Disk: 0
--- TESTING PREFECT/NOTEBOOK OCC SCAN TRIGGER ---
✅ Nextcloud OCC Scan completed via Zero-Trust Proxy!

🎉 Refresh your Nextcloud Web UI (http://192.168.1.55:30027). All new files are now visible!


In [12]:
# --- CONFIGURATION & MOUNTS ---
settings = load_settings()

mock_gdrive = repo_root / "data/mock/mock_gdrive"
mock_onedrive = repo_root / "data/mock/mock_onedrive"
mock_nextcloud = repo_root / "data/mock/mock_nextcloud"
real_nextcloud = Path("/nextcloud_data")
test_db = repo_root / "data/test_ledger.db"

cloud_sources = {"gdrive": str(mock_gdrive), "onedrive": str(mock_onedrive)}

print("🔐 Authenticating with Prefect Vault...")
llm_key = await Secret.load(settings["llm"]["secret_blocks"]["gemini_api_key"])
nc_user = await Secret.load(settings["nextcloud"]["secret_blocks"]["username"])
nc_pass = await Secret.load(settings["nextcloud"]["secret_blocks"]["password"])

llm_client = OpenAI(api_key=llm_key.get(), base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
webdav_client = Client(settings["nextcloud"]["url"], auth=(nc_user.get(), nc_pass.get()))

# --- 4. PHASE 0: ZFS-NATIVE RECONCILIATION TEST ---
print("\n" + "="*50)
print("🛡️  EXECUTING PHASE 0: RECONCILIATION")
print("="*50)

# A. Initialize v5 Schema with blake3_hash
init_schema(test_db, list(cloud_sources.keys()))

# B. Inject 1 fake ghost record (A path that physically DOES NOT exist on disk)
conn = sqlite3.connect(test_db)
conn.execute("""
    INSERT INTO production_inventory (blake3_hash, nextcloud_path, file_size)
    VALUES ('fake_ghost_hash_999', '/Documents/NonExistent_Ghost_File.pdf', 1234)
""")
conn.commit()
print("   - Injected fake ghost: /Documents/NonExistent_Ghost_File.pdf")
initial_count = conn.execute("SELECT COUNT(*) FROM production_inventory").fetchone()[0]
print(f"   - Production inventory count BEFORE reconciliation: {initial_count}")
conn.close()

# C. Run ZFS-Native Reconciliation against real disk existence
deleted_count = reconcile_deletions(test_db, real_nextcloud)
print(f"   - Reconciliation retired {deleted_count} ghost records.")

# D. Verify state
conn = sqlite3.connect(test_db)
final_count = conn.execute("SELECT COUNT(*) FROM production_inventory").fetchone()[0]
print(f"   - Production inventory count AFTER reconciliation: {final_count}")
conn.close()

print("   ✅ Phase 0 Reconciliation Complete!")


# --- EXECUTING ELT PIPELINE (PHASES 1-3) ---
print("\n" + "="*50)
print("🚀 EXECUTING ELT PIPELINE (PHASES 1-3)")
print("="*50)

# Phase 1: Scan sources into isolated staging tables (BLAKE3 Hashing)
scan_sources(test_db, cloud_sources)

# Phase 2: Cross-table dedupe & Taxonomy Strategy (SQLite Cache -> LLM Fallback)
routings = generate_strategy(test_db, list(cloud_sources.keys()), llm_client, settings["llm"]["model"])

# Phase 3: Physical Ingestion (Pre-flight Check + ZFS shutil.copy + Production Inventory Update)
execute_ingestion(test_db, real_nextcloud, routings)

print("--- TESTING PREFECT/NOTEBOOK OCC SCAN TRIGGER ---")

# Trigger the scan directly from Python
success = trigger_nextcloud_occ_scan(container_name="ix-nextcloud-nextcloud-1")

if success:
    print("\n🎉 Refresh your Nextcloud Web UI (http://192.168.1.55:30027). All new files are now visible!")

🔐 Authenticating with Prefect Vault...

🛡️  EXECUTING PHASE 0: RECONCILIATION
Schema initialized. Staging tables created for: ['gdrive', 'onedrive']
   - Injected fake ghost: /Documents/NonExistent_Ghost_File.pdf
   - Production inventory count BEFORE reconciliation: 6
🛡️  Starting Reconciliation Phase: Checking ZFS for manual deletions...
   🔥 Manual deletion detected! Purging ghost record: /Documents/NonExistent_Ghost_File.pdf
   🧹 Reconciliation Complete. Purged 1 ghost records from inventory.
   - Reconciliation retired 1 ghost records.
   - Production inventory count AFTER reconciliation: 5
   ✅ Phase 0 Reconciliation Complete!

🚀 EXECUTING ELT PIPELINE (PHASES 1-3)
🔍 Scanning source: gdrive at /home/coder/projects/agentic-nas-workflow/data/mock/mock_gdrive...
   ✅ gdrive: 5 files staged.
🔍 Scanning source: onedrive at /home/coder/projects/agentic-nas-workflow/data/mock/mock_onedrive...
   ✅ onedrive: 2 files staged.
✅ No new unique files to route. System is fully synced.

✅ Inges

In [11]:
import time
import pandas as pd
print("\n" + "="*50)
print("📊 PIPELINE EXECUTION TELEMETRY (from SQLite Ledger)")
print("="*50)

conn = sqlite3.connect(test_db)
conn.row_factory = sqlite3.Row

# Query the final state
scanned_count = conn.execute("SELECT COUNT(*) FROM staging_gdrive").fetchone()[0] + conn.execute("SELECT COUNT(*) FROM staging_onedrive").fetchone()[0]
uploaded_count = conn.execute("SELECT COUNT(*) FROM production_inventory").fetchone()[0]
duplicate_count = conn.execute("SELECT COUNT(*) FROM staging_gdrive WHERE status='duplicate'").fetchone()[0] + conn.execute("SELECT COUNT(*) FROM staging_onedrive WHERE status='duplicate'").fetchone()[0]
error_count = conn.execute("SELECT COUNT(*) FROM staging_gdrive WHERE status='error'").fetchone()[0] + conn.execute("SELECT COUNT(*) FROM staging_onedrive WHERE status='error'").fetchone()[0]
bytes_uploaded = conn.execute("SELECT SUM(file_size) FROM production_inventory").fetchone()[0] or 0
mb_uploaded = bytes_uploaded / (1024 * 1024)

# Print Telemetry
print(f"📁 Files Scanned:    {scanned_count}")
print(f"✅ Files Uploaded:   {uploaded_count} ({mb_uploaded:.2f} MB)")
print(f"🛑 Duplicates Saved: {duplicate_count}")
print(f"⚠️  Errors:           {error_count}")
print("="*50)

# Verify Physical State
print("\n🔍 Final Production Inventory:")
display(pd.read_sql_query("SELECT blake3_hash, nextcloud_path FROM production_inventory", conn))
print("\n🔍 Final OneDrive Staging State (Should show one duplicate):")
display(pd.read_sql_query("SELECT original_path, status FROM staging_onedrive", conn))

conn.close()


📊 PIPELINE EXECUTION TELEMETRY (from SQLite Ledger)
📁 Files Scanned:    7
✅ Files Uploaded:   5 (0.00 MB)
🛑 Duplicates Saved: 0
⚠️  Errors:           0

🔍 Final Production Inventory:


                                         blake3_hash  \
0  0b9043d29933a2aba055a82f3a31d083364f53de5a9565...   
1  42bda40860bf4172587d265575376d6bdd656e02fcefba...   
2  53f34198c1152cadab64c5745d9ecff100a9f1cf0aa961...   
3  f70b14b1869518d3cd5ff3844651ced3aba40cbbd48149...   
4  f9ea87c1bcce628cc6260f8d832dabe258857fd2513980...   

                                nextcloud_path  
0      /Documents/Reports/aws_invoice_july.txt  
1  /Documents/Financial/Taxes/family_photo.jpg  
2                 /Documents/Reports/photo.jpg  
3      /Documents/Financial/Taxes/tax_2026.pdf  
4                /Documents/Reports/report.pdf  


🔍 Final OneDrive Staging State (Should show one duplicate):


                                       original_path    status
0  /home/coder/projects/agentic-nas-workflow/data...  ingested
1  /home/coder/projects/agentic-nas-workflow/data...  ingested

### Step 4.4: Final Test with Prefect Pipeline

In [8]:
# Run the complete SRE ingestion and organization pipeline
from src.main import run_pipeline

In [ ]:
try:
    print("Initializing Agentic NAS Pipeline flow...")
    run_pipeline()
except OSError as e:
    print("\n[Environment Notice]: Flow execution completed with exception/warning.")
    print(f"Error detail: {e}")
    print("This is normal when Prefect or Nextcloud services are not fully running locally.")

Initializing Agentic NAS Pipeline flow...


00:10:35.418 | INFO    | Flow run 'metal-malamute' - Beginning flow run 'metal-malamute' for flow 'Agentic-NAS-Pipeline'

00:10:35.441 | INFO    | Flow run 'metal-malamute' - View at http://192.168.1.55:4200/runs/flow-run/4bc50acc-d2e8-43a8-b782-0e521da0dc9c

00:10:35.448 | INFO    | Flow run 'metal-malamute' - Starting SRE Pipeline on directory: /cloud_ingest/gdrive_ahabib9387

00:10:35.455 | INFO    | Flow run 'metal-malamute' - Authenticating with Prefect Vault...

...

00:14:12.947 | INFO    | Flow run 'metal-malamute' - 
Processing: Aurthohin - Ashte Sotto (music.com.bd).mp3

00:14:13.074 | INFO    | Task run 'task_hash_file-55e' - Finished in state Completed()

00:14:13.658 | INFO    | Task run 'task_upload_file-d29' -    [Success] Uploaded to -> /Auto_Organized/Aurthohin - Ashte Sotto (music.com.bd).mp3

00:14:13.665 | INFO    | Task run 'task_upload_file-d29' - Finished in state Completed()

00:14:14.683 | INFO    | Flow run 'metal-malamute' - Finished in state Completed()